# Group-split CNN baseline runner

This run fits only `NBRFI` versus `None` from the corrected group split. `NoneWNBRFI` remains an external hard-negative evaluation universe, so it cannot influence fitting, checkpoint selection, or the decision threshold.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'rfimt').is_dir():
            return candidate
    raise RuntimeError('Could not locate the rfimt repository above the notebook directory.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from rfimt.experiments import load_experiment_spec, make_run_manifest, write_run_manifest
from rfimt.metrics import best_threshold_by_f1, eval_binary
from rfimt.models import CNN1DRFI256Logits
from rfimt.training import ProfileDataset, collect_torch_scores

## Load the declared artifact set

The saved split indices, rather than a new random split, preserve the group-level separation established during dataset generation.

In [2]:
CONFIG_PATH = REPO_ROOT / 'configs' / 'experiments' / 'b0531_group_split_cnn_legacy_max_v1.json'
spec = load_experiment_spec(CONFIG_PATH)
dataset_spec = spec['dataset']
run_dir = Path(spec['outputs']['run_directory'])
if run_dir.exists():
    raise FileExistsError(f'Run directory already exists: {run_dir}. Use a new experiment id for a new run.')
run_dir.mkdir(parents=True)

torch.manual_seed(spec['model']['random_state'])
np.random.seed(spec['model']['random_state'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

meta = pd.read_csv(dataset_spec['metadata_path']).fillna('None')
profiles = np.load(dataset_spec['array_path'], mmap_mode='r')
splits = np.load(dataset_spec['split_indices_path'])

if len(meta) != len(profiles):
    raise ValueError('Subset metadata and profile array have different row counts.')
if set(meta['label'].unique()).difference({'NBRFI', 'None'}):
    raise ValueError('The fitting subset must contain only NBRFI and None rows.')

# Reuse the audited group split; do not resample rows in this runner.
split_indices = {name: np.asarray(splits[name], dtype=int) for name in ('train', 'val', 'test')}
y = meta['label'].eq(spec['labels']['positive']).to_numpy(dtype=np.float32)
print({name: len(indices) for name, indices in split_indices.items()}, device)

{'train': 16000, 'val': 2000, 'test': 2000} cpu


## Normalize and construct loaders

Per-channel z-scoring is applied before dataset construction so every profile uses its own mean and scale, without estimating parameters from validation or test rows.

In [3]:
normalization = spec['preprocessing']['normalization_variant']
if normalization == 'zscore_per_segment':
    raise ValueError('Use the dedicated z-score-per-segment runner; it prepares complete segments before sampling rows.')

# Each profile is scaled independently, so no fitted statistics cross split boundaries.
datasets = {
    name: ProfileDataset(np.asarray(profiles[indices]), y[indices], normalization=normalization)
    for name, indices in split_indices.items()
}
batch_size = spec['training']['batch_size']
loaders = {
    'train': DataLoader(datasets['train'], batch_size=batch_size, shuffle=True),
    'val': DataLoader(datasets['val'], batch_size=batch_size, shuffle=False),
    'test': DataLoader(datasets['test'], batch_size=batch_size, shuffle=False),
}

## Train and select the checkpoint

Validation loss selects the model independently of the F1 threshold. The threshold is chosen only after training from validation scores.

In [4]:
model = CNN1DRFI256Logits(**spec['model']['parameters']).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=spec['training']['learning_rate'])
checkpoint_path = run_dir / 'checkpoint.pt'
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

# Captured terminal notebooks render nested bars as thousands of log lines.
# Keep one sparse epoch-level indicator while batches run without their own bars.
for epoch in tqdm(range(spec['training']['epochs']), desc='Epochs', mininterval=5.0, miniters=5):
    model.train()
    train_loss_sum = 0.0
    train_count = 0
    for batch_profiles, batch_labels in loaders['train']:
        batch_profiles = batch_profiles.to(device)
        batch_labels = batch_labels.to(device).float().reshape(-1)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(batch_profiles), batch_labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(batch_labels)
        train_count += len(batch_labels)

    model.eval()
    val_loss_sum = 0.0
    val_count = 0
    with torch.no_grad():
        for batch_profiles, batch_labels in loaders['val']:
            batch_profiles = batch_profiles.to(device)
            batch_labels = batch_labels.to(device).float().reshape(-1)
            loss = loss_fn(model(batch_profiles), batch_labels)
            val_loss_sum += loss.item() * len(batch_labels)
            val_count += len(batch_labels)

    train_loss = train_loss_sum / train_count
    val_loss = val_loss_sum / val_count
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    # Checkpoint selection uses loss, leaving F1 threshold selection independent.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': epoch + 1, 'model_state_dict': model.state_dict(), 'val_loss': val_loss}, checkpoint_path)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], label='train')
ax.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], label='validation')
ax.set(xlabel='Epoch', ylabel='BCE loss')
ax.legend()
fig.tight_layout()
fig.savefig(run_dir / 'learning_curve.png', dpi=160)
plt.close(fig)

Epochs: 100%|██████████| 30/30 [02:08<00:00,  4.28s/it]


## Evaluate the selected model once

Reloading the checkpoint ensures reported test scores come from the minimum-validation-loss model. Segment scores are channel-score means; each segment must have one ground-truth label.

In [5]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Validation alone chooses the operating threshold; test remains untouched until now.
val_targets, val_scores = collect_torch_scores(model, loaders['val'], device=device)
if not np.array_equal(val_targets.astype(np.float32), y[split_indices['val']]):
    raise RuntimeError('Validation loader changed target order.')
threshold, validation_f1 = best_threshold_by_f1(y[split_indices['val']], val_scores)
test_targets, test_scores = collect_torch_scores(model, loaders['test'], device=device)
if not np.array_equal(test_targets.astype(np.float32), y[split_indices['test']]):
    raise RuntimeError('Test loader changed target order.')
row_test_metrics = eval_binary(y[split_indices['test']], test_scores, threshold=threshold)

test_frame = meta.iloc[split_indices['test']][['segment_index']].copy()
test_frame['label'] = y[split_indices['test']].astype(int)
test_frame['score'] = test_scores
test_frame['prediction'] = (test_frame['score'] >= threshold).astype(int)
segment_table = test_frame.groupby('segment_index', as_index=False).agg(label=('label', 'first'), score=('score', 'mean'), predicted_positive_fraction=('prediction', 'mean'), n_rows=('label', 'size'), label_count=('label', 'nunique'))
if not segment_table['label_count'].eq(1).all():
    raise ValueError('A test segment contains inconsistent channel labels.')
segment_metrics = eval_binary(segment_table['label'], segment_table['score'], threshold=threshold)
segment_table.to_csv(run_dir / 'segment_metrics.csv', index=False)

hard_negative_meta = pd.read_csv(dataset_spec['hard_negative_metadata_path']).fillna('None')
if not hard_negative_meta['label'].eq(spec['labels']['hard_negative']).all():
    raise ValueError('hard_negative_meta contains a label outside the declared hard-negative universe.')
full_source_array = np.load(dataset_spec['full_source_array_path'], mmap_mode='r')
# Hard negatives were excluded from the subset and are recovered from the immutable source array.
hard_negative_rows = full_source_array[hard_negative_meta['source_row_index'].to_numpy(dtype=int)]
hard_negative_dataset = ProfileDataset(np.asarray(hard_negative_rows), np.zeros(len(hard_negative_rows), dtype=np.float32), normalization=normalization)
hard_negative_loader = DataLoader(hard_negative_dataset, batch_size=batch_size, shuffle=False)
_, hard_negative_scores = collect_torch_scores(model, hard_negative_loader, device=device)
hard_negative_metrics = eval_binary(np.zeros(len(hard_negative_scores), dtype=int), hard_negative_scores, threshold=threshold)
hard_negative_metrics['n_rows'] = int(len(hard_negative_scores))
hard_negative_metrics['false_positive_rate'] = float((hard_negative_scores >= threshold).mean())
hard_negative_metrics_for_json = {
    key: (None if isinstance(value, float) and not np.isfinite(value) else value)
    for key, value in hard_negative_metrics.items()
}
with (run_dir / 'hard_negative_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(hard_negative_metrics_for_json, handle, indent=2, sort_keys=True, allow_nan=False)
    handle.write('\n')

## Record the run

The manifest stores the immutable experiment declaration, selected threshold, evaluation results, and artifact paths needed for comparison.

In [6]:
def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

code_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
manifest = make_run_manifest(
    spec,
    code_revision=code_revision,
    metrics=json_ready({
        'validation': {'threshold': threshold, 'f1_at_selected_threshold': validation_f1, 'checkpoint_val_loss': best_val_loss},
        'row_test': row_test_metrics,
        'segment_test': segment_metrics,
        'hard_negative': hard_negative_metrics,
    }),
    artifacts={
        'checkpoint': str(checkpoint_path),
        'learning_curve': str(run_dir / 'learning_curve.png'),
        'segment_metrics': str(run_dir / 'segment_metrics.csv'),
        'hard_negative_metrics': str(run_dir / 'hard_negative_metrics.json'),
    },
    notes=['Checkpoint selected by minimum validation loss; threshold selected by validation F1; test evaluated once.'],
)
write_run_manifest(run_dir / 'run_manifest.json', manifest)
print(f'Wrote run artifacts to {run_dir}')

Wrote run artifacts to /hercules/results/akazantsev/rfim_dataset/runs/b0531_group_split_cnn_legacy_max_v1
